# TD 6 — Élan et surapprentissage : lire les courbes d'apprentissage

## Objectifs

À la fin de la séance, vous saurez :

- distinguer l'**erreur d'optimisation** (coût d'entraînement) de l'**écart train/test** (capacité à généraliser) ;
- reconnaître les **régimes** d'une courbe d'apprentissage : descente, plateau, remontée (surapprentissage), divergence ;
- **prédire puis mesurer** l'effet du coefficient d'élan $\mu$ sur la dynamique d'apprentissage ;
- distinguer un **zigzag systématique** (causé par l'élan) d'un **bruit de batch** (causé par les mini-lots).

## Plan de séance (≈ 3 h)

1. **Ex. 1** — Premier contact : comparaison des courbes train/test avec et sans élan sur le jeu 2D standard.
2. **Ex. 2** — La vallée mal conditionnée : zigzag, accélération ×10, divergence à grand pas.
3. **Ex. 3** — Le bruit mémorisé : élan et surapprentissage.
4. **Ex. 4** (optionnel) — Mini-batchs : l'élan lisse-t-il vraiment ?
5. **Bilan** — synthèse et règles de lecture des courbes.

## Rappel : règle de mise à jour avec élan (momentum)

$$v_l \leftarrow \mu\, v_l + \eta\, \frac{\partial E}{\partial W_l}, \qquad W_l \leftarrow W_l - v_l$$

La **vitesse** $v_l$ accumule les gradients passés, pondérés par le facteur d'élan $\mu \in [0, 1[$ : le pas effectif est plus grand quand le gradient garde le même sens, et les oscillations sont amorties.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy

# Alias de types pour les tableaux numpy
Vec = np.ndarray[tuple[int], np.dtype[np.float64]]        # vecteur 1D de flottants
Mat = np.ndarray[tuple[int, int], np.dtype[np.float64]]   # matrice 2D de flottants

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_prime(z: Mat) -> Mat:
    """
    σ'(z) = σ(z) · (1 - σ(z)).
    """
    s = sigmoid(z)
    return s * (1.0 - s)

def tanh_prime(z: Mat) -> Mat:
    """
    tanh'(z) = 1 - tanh²(z).
    """
    return 1.0 - np.tanh(z) ** 2

ACTIVATIONS = {
    "sigmoid": (sigmoid, sigmoid_prime),
    "tanh": (np.tanh, tanh_prime),
    "linear": (lambda z: z, lambda z: np.ones_like(z)),
}

In [ ]:
class MLP:
    def __init__(self, sizes, activation="sigmoid", output_activation="linear", seed=0):
        rng = np.random.default_rng(seed)
        self.sizes = list(sizes)
        self.Ws = []
        self.bs = []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            self.Ws.append(rng.normal(0.0, 0.5, size=(n_out, n_in)))
            self.bs.append(np.zeros(n_out))
        self.activation, self.activation_prime = ACTIVATIONS[activation]
        self.output_activation, self.output_activation_prime = ACTIVATIONS[output_activation]

    def predict(self, X):
        return self.forward(X)[0][-1]

    def loss(self, X: Mat, t: Mat) -> float:
        """Coût du modèle : erreur quadratique moyenne.

        E = (1/2) · mean(Σ_k (Y_k - T_k)²).

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).

        Returns
        -------
        float
            Coût quadratique moyen sur le batch.
        """
        # coût du cours : moyenne sur les échantillons de 1/2 · Σ_k (Y_k - T_k)^2
        return 0.5 * float(np.mean(np.sum((self.predict(X) - t) ** 2, axis=1)))

    def forward(self, X: Mat) -> tuple[list[Mat], list[Mat]]:
        """Passe avant du MLP, couche par couche.

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).

        Returns
        -------
        tuple[list[Mat], list[Mat]]
            (As, Zs) : activations (avec Y0 = X en tête) et pré-activations
            de chaque couche.
        """
        A = X
        As = [A]
        Zs = []
        n = len(self.Ws)
        for l, (W, b) in enumerate(zip(self.Ws, self.bs)):
            Z = A @ W.T + b
            Zs.append(Z)
            f = self.output_activation if l == n - 1 else self.activation
            A = f(Z)
            As.append(A)
        return As, Zs

    def backward(self, X: Mat, t: Mat) -> list[tuple[Mat, Vec]]:
        """Passe arrière : gradients du coût pour chaque couche.

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).

        Returns
        -------
        list[tuple[Mat, Vec]]
            Liste de (dW_l, db_l), une paire de gradients par couche.
        """
        As, Zs = self.forward(X)
        N = X.shape[0]
        n = len(self.Ws)
        delta = (As[-1] - t) * self.output_activation_prime(Zs[-1])
        dWs, dbs = [], []
        for l in range(n - 1, -1, -1):
            dW = delta.T @ As[l] / N
            db = delta.sum(axis=0) / N
            dWs.insert(0, dW)
            dbs.insert(0, db)
            if l > 0:
                delta = (delta @ self.Ws[l]) * self.activation_prime(Zs[l - 1])
        return list(zip(dWs, dbs))

    def step(self, X: Mat, t: Mat, lr: float) -> None:
        """Effectue une étape de descente de gradient (mise à jour des poids).

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).
        lr : float
            Pas d'apprentissage.
        """
        for l, (dW, db) in enumerate(self.backward(X, t)):
            self.Ws[l] -= lr * dW
            self.bs[l] -= lr * db

    def fit_batch(self, X: Mat, t: Mat, X_test: Mat | None = None, t_test: Mat | None = None, lr: float = 0.1, epochs: int = 2000) -> dict[str, list[float]]:
        """Entraîne le MLP par descente de gradient par lot.

        Si ``X_test``/``t_test`` sont fournis, le coût de test est aussi
        ajouté à l'historique.

        Parameters
        ----------
        X : Mat
            Batch d'entraînement, de forme (N, d).
        t : Mat
            Cibles d'entraînement, de forme (N, k).
        X_test : Mat, optionnel
            Batch de test, de forme (N_test, d).
        t_test : Mat, optionnel
            Cibles de test, de forme (N_test, k).
        lr : float, optionnel
            Pas d'apprentissage (défaut : 0.1).
        epochs : int, optionnel
            Nombre maximal d'epochs (défaut : 2000).

        Returns
        -------
        dict[str, list[float]]
            Historique des coûts ``{"train": [...], "test": [...]}``.
        """
        history = {"train": []}
        if X_test is not None:
            history["test"] = []
        for epoch in range(epochs):
            self.step(X, t, lr)
            history["train"].append(self.loss(X, t))
            if X_test is not None:
                history["test"].append(self.loss(X_test, t_test))
        return history


In [ ]:
class MLPMomentum(MLP):
    def __init__(self, sizes, activation="sigmoid", output_activation="linear", seed=0, momentum=0.9):
        super().__init__(sizes, activation, output_activation, seed)
        self.momentum = momentum
        self.vWs = [np.zeros_like(W) for W in self.Ws]
        self.vbs = [np.zeros_like(b) for b in self.bs]

    def step(self, X: Mat, t: Mat, lr: float) -> None:
        """Une étape de descente de gradient avec élan (momentum).

        v_l <- mu * v_l + lr * dW_l ; W_l <- W_l - v_l.
        """
        for l, (dW, db) in enumerate(self.backward(X, t)):
            self.vWs[l] = self.momentum * self.vWs[l] + lr * dW
            self.vbs[l] = self.momentum * self.vbs[l] + lr * db
            self.Ws[l] -= self.vWs[l]
            self.bs[l] -= self.vbs[l]


In [ ]:
def gen_2d(n=500, noise=0.1, seed=1):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-3.0, 3.0, size=(n, 2))
    t = (np.sin(X[:, 0]) * np.cos(X[:, 1])).reshape(-1, 1) + rng.normal(0.0, noise, size=(n, 1))
    return X, t

def plot_history(history, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    ax.plot(history["train"], label="train")
    if history["test"]:
        ax.plot(history["test"], label="test")
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE")
    ax.set_yscale("log")
    ax.legend()
    ax.grid(True)
    return ax

def split(X: Mat, t: Mat, n_train: int, seed: int = 0) -> tuple[Mat, Mat, Mat, Mat]:
    """Sépare aléatoirement un jeu de données en entraînement et test.

    Les échantillons sont mélangés par une permutation déterministe
    (graine ``seed``), puis découpés : les ``n_train`` premiers forment
    l'ensemble d'entraînement, les suivants l'ensemble de test.

    Parameters
    ----------
    X : Mat
        Données d'entrée, de forme (N, d).
    t : Mat
        Cibles, de même forme que la sortie attendue (N, k).
    n_train : int
        Taille de l'ensemble d'entraînement.
    seed : int, optionnel
        Graine du générateur aléatoire (défaut : 0).

    Returns
    -------
    tuple[Mat, Mat, Mat, Mat]
        ``(X_train, t_train, X_test, t_test)``.
    """
    rng = np.random.default_rng(seed)
    perm = rng.permutation(X.shape[0])
    X_shuf, t_shuf = X[perm], t[perm]
    return X_shuf[:n_train], t_shuf[:n_train], X_shuf[n_train:], t_shuf[n_train:]


## Exercice 1 — Premier contact : avec/sans élan

### Énoncé

On reprend le jeu 2D standard du TD03 : cible $t = \sin(x_1)\cos(x_2)$, avec un bruit gaussien d'écart-type $\sigma = 0.1$ ; après découpage, **400** échantillons d'entraînement et **100** de test.

Architecture et conditions d'entraînement, **identiques** pour les deux variantes : réseau `[2, 16, 16, 1]`, tanh en caché, sortie **linéaire**, même graine, `lr = 0.2`, `epochs = 20000`.

> **Pronostic (a) — tempo train** — Avant tout entraînement : quel effet attendez-vous de $\mu = 0.9$ sur la vitesse de descente de la loss d'entraînement ?
>
> **Pronostic (b) — courbe test** — Et la courbe test : que fera chaque variante au fil des epochs (plateau ? remontée ?) ?
>
> *Notez vos pronostics : vous les confronterez aux courbes de la cellule suivante.*

### Corrigé

> **Exercice 1** — Générez le jeu 2D (bruit $\sigma = 0.1$), découpez-le avec `split` (400 en train, graine 1), puis entraînez la variante sans élan (`MLP`) et avec élan (`MLPMomentum`, $\mu = 0.9$). Tracez les deux historiques côte à côte (échelle log).


In [ ]:
# BEGIN
X4, t4 = gen_2d(noise=0.1)
X_train, t_train, X_test, t_test = split(X4, t4, n_train=400, seed=1)

lr, epochs = 0.2, 20000

model_plain = MLP([2, 16, 16, 1], activation="tanh", output_activation="linear", seed=0)
hist_plain = model_plain.fit_batch(X_train, t_train, X_test, t_test, lr=lr, epochs=epochs)

model_mom = MLPMomentum([2, 16, 16, 1], activation="tanh", output_activation="linear", seed=0, momentum=0.9)
hist_mom = model_mom.fit_batch(X_train, t_train, X_test, t_test, lr=lr, epochs=epochs)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
plot_history(hist_plain, axes[0])
plot_history(hist_mom, axes[1])
axes[0].set_title("sans élan (μ=0)")
axes[1].set_title("avec élan (μ=0.9)")
# END


### Lecture guidée du corrigé

Comparez avec vos pronostics. Les chiffres ci-dessous se lisent **après coup** sur la figure — l'énoncé ne les donnait pas à l'avance.

- **Train** : les deux courbes convergent de façon quasiment identique (valeurs finales ≈ 0.0050 sans élan, ≈ 0.0042 avec élan). Sur ce jeu, l'élan ne change presque rien à l'optimisation du train.
- **Test, sans élan** : la loss descend vers un plateau à ≈ 0.0068, atteint dans le dernier quart de l'entraînement, puis y reste. Aucune remontée : sur ce jeu riche (400 exemples) et bruité modérément, il n'y a tout simplement pas de surapprentissage.
- **Test, avec élan** : minimum ≈ 0.0078 atteint très tôt (vers l'epoch 3 549), puis remontée lente jusqu'à ≈ 0.0109 : les pas effectifs plus grands font que le réseau continue à progresser sur le train là où la version sans élan s'était stabilisée — le début d'un surapprentissage que l'élan accélère.

> **Phrase à retenir** — Sur ce jeu, l'élan ne dégrade pas grand-chose, mais il **fait bouger** la courbe test — les exercices suivants amplifient ce phénomène.


## Exercice 2 — La vallée mal conditionnée

### Énoncé

Le générateur `gen_vallee` (fourni dans le corrigé) produit $n = 200$ points :

- $x_1 \sim \mathcal{U}(-1,\,1)$ et $x_2 \sim \mathcal{U}(-80,\,80)$ : deux axes d'échelles très différentes ;
- cible $t = 0.5\,x_1 + 0.02\,x_2 + \varepsilon$, avec $\varepsilon \sim \mathcal{N}(0,\,0.05^2)$ ;
- **aucune normalisation des entrées**.

Le réseau est **linéaire** : `[2, 1]` (une seule couche, qui est la couche de sortie, activation de
sortie linéaire) — un poids par axe, pas de non-linéarité. Les deux variantes partent de la même
graine.

> **Exercice 2 (a) — pronostic** — Avant tout entraînement : avec `lr = 7e-4`, que feront les
> variantes **sans** puis **avec** élan ($\mu = 0.9$) ? Laquelle convergera, et à quel **tempo
> relatif** sur le train ?
>
> **Exercice 2 (b) — chronomètre** — « Plus vite » se mesure mieux par le **nombre d'epochs pour
> passer sous un seuil** de loss que par la valeur à l'epoch finale. Écrivez
> `epochs_to_threshold(hist, thr=0.005)` : le premier index de l'historique d'entraînement où la
> loss passe sous `thr` (`None` sinon).
>
> **Exercice 2 (c) — monter le pas** — Passez `lr = 1.3e-3` et recommencez les deux variantes :
> que fait la variante sans élan ? et celle avec élan ?
>
> **Exercice 2 (d) — option corrigée** — Visualisez la trajectoire des poids $(w_1, w_2)$ des deux
> variantes par-dessus le paysage de perte du réseau linéaire : que fait chaque variante dans la
> vallée ?
>
> **Exercice 2 (e) — discussion** — Que changerait la **standardisation des entrées** ?
> Formulez votre réponse avant de lire le test rapide en fin d'exercice.

### Corrigé — partie 1

> **Exercice 2** — Générez la vallée avec `gen_vallee()` (`n=200`, `scale=80.0`, `noise=0.05`,
> `seed=3`). Entraînez le réseau `[2, 1]` **sans élan** (`MLP`) puis **avec élan**
> (`MLPMomentum`, $\mu = 0.9$), même graine `seed=0`, `lr = 7e-4`, 6000 epochs ; chronométrez
> les deux runs avec `epochs_to_threshold` et tracez les historiques côte à côte (échelle log,
> seuil partagé à $0.005$).
>
> Puis recommencez à `lr = 1.3e-3` sur 2000 epochs : observez ce que fait chaque variante.


In [ ]:
# BEGIN
def gen_vallee(n=200, scale=80.0, noise=0.05, seed=3):
    """Vallée mal conditionnée : deux axes d'échelles très différentes.

    x1 ~ U(-1, 1), x2 ~ U(-scale, scale) ; t = 0.5·x1 + 0.02·x2 + ε ~ N(0, noise²).
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(-1.0, 1.0, size=n)
    x2 = rng.uniform(-scale, scale, size=n)
    t = 0.5 * x1 + 0.02 * x2 + rng.normal(0.0, noise, size=n)
    return np.column_stack([x1, x2]), t.reshape(-1, 1)


def epochs_to_threshold(hist, thr=0.005):
    """Premier index où la loss d'entraînement passe sous ``thr`` (None sinon)."""
    for i, loss_train in enumerate(hist["train"]):
        if loss_train < thr:
            return i
    return None


lr, epochs = 7e-4, 6000
X_v, t_v = gen_vallee()

model_plain = MLP([2, 1], output_activation="linear", seed=0)
hist_plain_v = model_plain.fit_batch(X_v, t_v, lr=lr, epochs=epochs)

model_mom = MLPMomentum([2, 1], output_activation="linear", seed=0, momentum=0.9)
hist_mom_v = model_mom.fit_batch(X_v, t_v, lr=lr, epochs=epochs)

ep_plain_v = epochs_to_threshold(hist_plain_v)
ep_mom_v = epochs_to_threshold(hist_mom_v)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, hist, ep, titre, annot in (
    (axes[0], hist_plain_v, ep_plain_v, "sans élan (μ=0)", (ep_plain_v - 150, 0.2, "right")),
    (axes[1], hist_mom_v, ep_mom_v, "avec élan (μ=0.9)", (ep_mom_v + 450, 0.2, "left")),
):
    ax.plot(hist["train"])
    ax.axhline(0.005, color="crimson", ls=":", lw=1)
    if ep is not None:
        ax.axvline(ep, color="crimson", ls="--", lw=1)
        ax.annotate(f"seuil 0.005 : epoch {ep}", xy=(ep, 0.005),
                    xytext=annot[:2], ha=annot[2], color="crimson",
                    arrowprops=dict(arrowstyle="->", color="crimson"))
    ax.set_yscale("log")
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE (train)")
    ax.set_title(titre)
    ax.grid(True)
# END


In [ ]:
# BEGIN
lr_hi, epochs_hi = 1.3e-3, 2000

# le run sans élan va diverger (pertes non finies ou très grandes) : on coupe les avertissements
with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
    hist_plain_hi = MLP([2, 1], output_activation="linear", seed=0).fit_batch(
        X_v, t_v, lr=lr_hi, epochs=epochs_hi)

hist_mom_hi = MLPMomentum([2, 1], output_activation="linear", seed=0, momentum=0.9).fit_batch(
    X_v, t_v, lr=lr_hi, epochs=epochs_hi)

# pertes sans élan : non finies ou > 50 -> np.nan (trous dans la ligne)
plain_hi = np.array([
    v if np.isfinite(v) and v <= 50.0 else np.nan
    for v in hist_plain_hi["train"]
])
a_diverge = bool(np.isnan(plain_hi).any())
ep_mom_hi = epochs_to_threshold(hist_mom_hi)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].plot(plain_hi, ".-", ms=3)  # nan = trous : la courbe sort du cadre
axes[0].set_xlim(0, epochs_hi)
axes[0].set_title("sans élan (μ=0) : divergence")
axes[1].plot(hist_mom_hi["train"])
axes[1].set_title("avec élan (μ=0.9) : converge")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE (train)")
    ax.grid(True)

if a_diverge:
    premier_div = int(np.argmax(np.isnan(plain_hi)))
    axes[0].axvline(premier_div, color="crimson", ls="--", lw=1)
    axes[0].text(premier_div + 40, 12.0,
                 f"perte > 50 dès l'epoch {premier_div}\n"
                 "(au-delà : hors du cadre, la loss explose)",
                 color="crimson", va="top")
if ep_mom_hi is not None:
    axes[1].axvline(ep_mom_hi, color="green", ls="--", lw=1)
    axes[1].text(ep_mom_hi + 60, 1.5,
                 f"seuil atteint : epoch {ep_mom_hi}",
                 color="green", va="top")
# END


### Réponses attendues

- **(a)** Le run **sans élan** finit par converger, mais il **zigzague systématiquement sur l'axe
  raide** $w_2$ (celui des grandes valeurs de $x_2$) tout en rampant le long de l'axe facile
  $w_1$ : il met **4 858 epochs** à passer sous $0.005$. La variante **avec élan** fait dix fois
  mieux (**485 epochs**) : la vitesse $v \leftarrow \mu v + \eta\,\nabla E$ **moyenne les
  gradients** — les composantes oscillantes de l'axe raide se compensent d'un coup sur l'autre,
  celles de l'axe facile s'accumulent — ce qui accélère le long de l'axe facile et amortit le
  zigzag.
- **(b)** `epochs_to_threshold` mesure le premier passage sous le seuil : sans élan, des
  **milliers** d'epochs (4 858 sur nos runs) ; avec élan, des **centaines** (485). Les
  annotations des figures affichent les valeurs mesurées sur les runs — l'énoncé ne les donnait
  pas à l'avance.
- **(c)** À `lr = 1.3e-3`, le **sans élan diverge** : le pas dépasse la limite de stabilité de
  l'axe raide, les pertes explosent vers `inf`/`nan` dès les premières epochs. L'**élan amortit**
  ce dépassement (bref dépassement visible sur $w_2$) et converge : seuil atteint vers
  l'**epoch 261**.
- **(d)** et **(e)** : les corrigés des deux options sont ci-dessous (trajectoire des poids, puis
  standardisation).


### Corrigé (option) — la trajectoire des poids

Paysage du modèle linéaire sans biais :

$$E(w) = \tfrac{1}{2}\,\mathrm{mean}\big((X w - t)^2\big)$$

tracé sur la grille $w_1 \in [-0.65,\,0.65] \times w_2 \in [-0.09,\,0.09]$, avec les
trajectoires des poids des deux variantes (`lr = 7e-4`, 2500 epochs, enregistrement des poids tous
les 20 epochs via une boucle manuelle sur `model.step(X_v, t_v, lr)`). Départ : le $w$ initial
aléatoire (graine 0) ; étoile verte : l'optimum théorique $(0.5,\ 0.02)$. Avec `momentum=0`,
`MLPMomentum` redonne exactement la descente sans élan.


In [ ]:
# BEGIN
def paysage_vallee(X, t, w1s, w2s):
    """Paysage du modèle linéaire sans biais : E(w) = 0.5 · mean((X·w − t)²)."""
    E = np.empty((w2s.size, w1s.size))
    for j, w1 in enumerate(w1s):
        for k, w2 in enumerate(w2s):
            E[k, j] = 0.5 * np.mean((w1 * X[:, 0] + w2 * X[:, 1] - t[:, 0]) ** 2)
    return E


def trajectoire_poids(momentum, lr=7e-4, epochs=2500, pas=20, seed=0):
    """Entraîne le réseau [2, 1] et enregistre (w1, w2) tous les ``pas`` epochs."""
    model = MLPMomentum([2, 1], output_activation="linear", seed=seed, momentum=momentum)
    traj = [model.Ws[0].ravel().copy()]  # départ : w initial aléatoire
    for epoch in range(epochs):
        model.step(X_v, t_v, lr)
        if (epoch + 1) % pas == 0:
            traj.append(model.Ws[0].ravel().copy())
    traj.append(model.Ws[0].ravel().copy())
    return np.array(traj)


w1s = np.linspace(-0.65, 0.65, 201)
w2s = np.linspace(-0.09, 0.09, 201)
E = paysage_vallee(X_v, t_v, w1s, w2s)

traj_plain_p = trajectoire_poids(0.0)
traj_mom_p = trajectoire_poids(0.9)
opt = (0.5, 0.02)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for ax, traj, titre in (
    (axes[0], traj_plain_p, "sans élan (μ=0)"),
    (axes[1], traj_mom_p, "avec élan (μ=0.9)"),
):
    cf = ax.contourf(w1s, w2s, E, levels=40, cmap="Blues")
    ax.plot(traj[:, 0], traj[:, 1], ".-", color="crimson", lw=1, ms=3)
    ax.plot(traj[0, 0], traj[0, 1], "o", color="black", ms=6, label="départ (w initial)")
    ax.plot(*opt, "*", color="limegreen", ms=14, label="optimum (0.5, 0.02)")
    ax.set_xlabel("$w_1$ (poids de $x_1$)")
    ax.set_ylabel("$w_2$ (poids de $x_2$)")
    ax.set_title(titre)
    ax.legend(loc="upper left", fontsize="small")
fig.colorbar(cf, ax=axes, label="E(w)")
# END


### Pour aller plus loin (corrigé) — et la standardisation ?

Réflexion : standardiser les entrées ($x_j \leadsto (x_j - \bar{x}_j)\, /\, \sigma_j$) met les
deux axes à la même échelle. Le malconditionnement — le rapport des courbures de l'ordre de
6400 entre les deux axes — s'évanouit : on peut relever le `lr` sans divergence, la descente
avance à la même vitesse dans toutes les directions et **l'écart dramatique de tempo que l'élan
compensait disparaît** (l'élan redevient accessoire pour ce cas).

Vérifions-le : même réseau, même cible, entrées standardisées, `lr` relevé de deux ordres de
grandeur.


In [ ]:
# BEGIN
x1_std = (X_v[:, 0] - X_v[:, 0].mean()) / X_v[:, 0].std()
x2_std = (X_v[:, 1] - X_v[:, 1].mean()) / X_v[:, 1].std()
X_std = np.column_stack([x1_std, x2_std])

lr_std = 0.1  # impossible avant standardisation : l'axe raide diverge dès ~1e-3

hist_std_plain = MLP([2, 1], output_activation="linear", seed=0).fit_batch(
    X_std, t_v, lr=lr_std, epochs=6000)
hist_std_mom = MLPMomentum([2, 1], output_activation="linear", seed=0, momentum=0.9).fit_batch(
    X_std, t_v, lr=lr_std, epochs=6000)

print("standardisé, lr = 0.1 : seuil 0.005 atteint à l'epoch",
      epochs_to_threshold(hist_std_plain), "sans élan vs",
      epochs_to_threshold(hist_std_mom), "avec élan")
# END


Passons à l'exercice 3.